In [9]:
from scipy import sparse
import numpy as np
import pandas as pd
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report, f1_score
from sklearn.utils.class_weight import compute_class_weight

tfidf = joblib.load("../data/tfidf.pkl")
X_train = sparse.load_npz("../data/X_train_tfidf.npz")
X_test = sparse.load_npz("../data/X_test_tfidf.npz")
y_train = np.load("../data/y_train.npy")
y_test = np.load("../data/y_test.npy")
label_names = np.load("../data/label_names.npy", allow_pickle=True)

# Convert y arrays to DataFrame for convenience
y_train = pd.DataFrame(y_train, columns=label_names)
y_test = pd.DataFrame(y_test, columns=label_names)


# Logistic Regression with Class Weights Attempt

In [11]:
# Compute class weights per label
class_weights = {}

for label in label_names:
    weights = compute_class_weight(
        class_weight='balanced',
        classes=np.array([0, 1]),
        y=y_train[label]
    )
    class_weights[label] = {0: weights[0], 1: weights[1]}

# Create a separate logistic regression model for each label with its weights
models = {}

for label in label_names:
    w = class_weights[label]
    clf = LogisticRegression(
        class_weight=w,
        max_iter=2000,
        n_jobs=-1
    )
    clf.fit(X_train, y_train[label])
    models[label] = clf

# Predict
y_pred = np.column_stack([
    models[label].predict(X_test)
    for label in label_names
])

y_pred = pd.DataFrame(y_pred, columns=label_names)
# Evaluate
print(classification_report(y_test, y_pred, target_names=label_names, digits=4))


               precision    recall  f1-score   support

        toxic     0.6448    0.8482    0.7326      3056
 severe_toxic     0.2701    0.8287    0.4074       321
      obscene     0.6879    0.8805    0.7724      1715
       threat     0.1696    0.7838    0.2788        74
       insult     0.5630    0.8550    0.6790      1614
identity_hate     0.2314    0.7415    0.3528       294

    micro avg     0.5509    0.8516    0.6690      7074
    macro avg     0.4278    0.8229    0.5372      7074
 weighted avg     0.5974    0.8516    0.6947      7074
  samples avg     0.0612    0.0811    0.0666      7074



c:\Users\Yasna\Desktop\CMD Project 1\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Yasna\Desktop\CMD Project 1\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Yasna\Desktop\CMD Project 1\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} 

# LinearSVC (OneVsRest) Attempt

In [12]:
svm_model = OneVsRestClassifier(
    LinearSVC()
)

svm_model.fit(X_train, y_train)


,estimator,LinearSVC()
,n_jobs,None
,verbose,0
,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1


In [14]:
y_pred_svm = svm_model.predict(X_test)

print("Macro F1-score:", f1_score(y_test, y_pred_svm, average="macro"))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_svm, target_names=label_names, digits=4))

Macro F1-score: 0.5271388561187131

Classification Report:

               precision    recall  f1-score   support

        toxic     0.8569    0.6859    0.7619      3056
 severe_toxic     0.4932    0.2243    0.3084       321
      obscene     0.8823    0.6991    0.7801      1715
       threat     0.4688    0.2027    0.2830        74
       insult     0.7869    0.5675    0.6595      1614
identity_hate     0.6981    0.2517    0.3700       294

    micro avg     0.8323    0.6180    0.7093      7074
    macro avg     0.6977    0.4385    0.5271      7074
 weighted avg     0.8199    0.6180    0.7011      7074
  samples avg     0.0618    0.0555    0.0561      7074



c:\Users\Yasna\Desktop\CMD Project 1\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Yasna\Desktop\CMD Project 1\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Yasna\Desktop\CMD Project 1\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} 

# Calibrated LinearSVC (CalibratedClassifierCV wrap) Attempt

In [15]:
base_svm = LinearSVC()

calibrated_svm = OneVsRestClassifier(
    CalibratedClassifierCV(base_svm, cv=3, method="sigmoid")
)

calibrated_svm.fit(X_train, y_train)

y_pred_cal = calibrated_svm.predict(X_test)

print("Macro F1-score:", f1_score(y_test, y_pred_cal, average="macro"))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_cal, target_names=label_names, digits=4))

Macro F1-score: 0.5247190976545147

Classification Report:

               precision    recall  f1-score   support

        toxic     0.8811    0.6643    0.7575      3056
 severe_toxic     0.5431    0.1963    0.2883       321
      obscene     0.8966    0.6875    0.7782      1715
       threat     0.5455    0.2432    0.3364        74
       insult     0.8055    0.5310    0.6400      1614
identity_hate     0.7010    0.2313    0.3478       294

    micro avg     0.8551    0.5958    0.7023      7074
    macro avg     0.7288    0.4256    0.5247      7074
 weighted avg     0.8412    0.5958    0.6930      7074
  samples avg     0.0600    0.0527    0.0538      7074



c:\Users\Yasna\Desktop\CMD Project 1\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Yasna\Desktop\CMD Project 1\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Yasna\Desktop\CMD Project 1\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} 